# AutoDL 实验运行 Notebook

在 AutoDL JupyterLab 里逐步运行所有实验。

> **使用前提**：已通过 scp / JupyterLab 上传 `data/billsum/` 和 `data/casehold/`

## 0. 环境准备

In [4]:
import os
os.chdir('/root/MLP')
os.environ['HF_HOME'] = '/root/autodl-tmp/hf_cache'
os.environ['TRANSFORMERS_CACHE'] = '/root/autodl-tmp/hf_cache'
os.environ['HF_ENDPOINT'] = 'https://hf-mirror.com'
os.makedirs('/root/autodl-tmp/hf_cache', exist_ok=True)
os.makedirs('/root/autodl-tmp/outputs', exist_ok=True)
os.makedirs('/root/autodl-tmp/logs', exist_ok=True)
os.environ['PYTORCH_ALLOC_CONF'] = 'expandable_segments:True'
# 把 outputs/logs 软链到数据盘
if not os.path.islink('/root/MLP/outputs'):
    os.system('ln -sfn /root/autodl-tmp/outputs /root/MLP/outputs')
if not os.path.islink('/root/MLP/logs'):
    os.system('ln -sfn /root/autodl-tmp/logs /root/MLP/logs')

!pwd && ls
!df -h /root | tail -1
print(f"HF_HOME: {os.environ['HF_HOME']}")
!pwd
!ls


/root/MLP
README.md  autodl_run.ipynb  configs  data  logs  outputs  scripts  src  wandb
overlay          30G   26G  4.5G  86% /
HF_HOME: /root/autodl-tmp/hf_cache
/root/MLP
README.md  autodl_run.ipynb  configs  data  logs  outputs  scripts  src  wandb


In [ ]:
import sys
print(f"当前 kernel Python: {sys.executable}")
!{sys.executable} -m pip install peft accelerate trl bitsandbytes wandb rouge-score bert-score scikit-learn sentencepiece huggingface_hub
print("\n✅ 安装完成！请立即重启内核再继续：菜单 Kernel → Restart Kernel")


In [5]:
import torch, peft, trl
print('torch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'None')


torch: 2.10.0+cu128
CUDA: True
GPU: NVIDIA H800 PCIe


In [6]:
# 验证数据文件
import os
files = [
    'data/billsum/train_sft.jsonl',
    'data/billsum/val_sft.jsonl',
    'data/billsum/test_us_sft.jsonl',
    'data/billsum/test_ca_sft.jsonl',
    'data/casehold/train_mc.jsonl',
    'data/casehold/validation_mc.jsonl',
    'data/casehold/test_mc.jsonl',
]
for f in files:
    size = os.path.getsize(f) // 1024 if os.path.exists(f) else -1
    status = f'✓ {size} KB' if size >= 0 else '✗ 缺失！'
    print(f'{status}  {f}')

✓ 282658 KB  data/billsum/train_sft.jsonl
✓ 15130 KB  data/billsum/val_sft.jsonl
✓ 51177 KB  data/billsum/test_us_sft.jsonl
✓ 12753 KB  data/billsum/test_ca_sft.jsonl
✓ 115712 KB  data/casehold/train_mc.jsonl
✓ 19727 KB  data/casehold/validation_mc.jsonl
✓ 19716 KB  data/casehold/test_mc.jsonl


## 1. BillSum LoRA — Qwen

In [8]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
os.makedirs('logs', exist_ok=True)
!{sys.executable} -u src/train/train.py --config configs/lora_billsum_qwen.yaml 2>&1 | tee logs/lora_billsum_qwen.log
print("训练完成")


/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/lora_billsum_qwen.yaml
Model   : Qwen/Qwen2.5-1.5B-Instruct
Train   : /root/MLP/data/billsum/train_sft.jsonl  (15,988 records)
Val     : /root/MLP/data/billsum/val_sft.jsonl
Max input length : 2048
Max output length: 512
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 602.93it/s]
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

Preparing datasets...
Tokenizing: 100%|████

In [19]:
import sys, os
os.chdir('/root/MLP')
!{sys.executable} -u src/evaluate/inference.py --config configs/lora_billsum_qwen.yaml --split test_us
!{sys.executable} -u src/evaluate/inference.py --config configs/lora_billsum_qwen.yaml --split test_ca
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/lora_billsum_qwen/predictions_test_us.jsonl --output outputs/lora_billsum_qwen/eval_test_us.json
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/lora_billsum_qwen/predictions_test_ca.jsonl --output outputs/lora_billsum_qwen/eval_test_ca.json
print("评估完成")
!cat outputs/lora_billsum_qwen/eval_test_us.json


Config     : configs/lora_billsum_qwen.yaml
Task       : summarization
Split      : test_us  →  /root/MLP/data/billsum/test_us_sft.jsonl
Output     : /root/MLP/outputs/lora_billsum_qwen
Batch size : 8
/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
Loading base model: Qwen/Qwen2.5-1.5B-Instruct
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|███████████████████████| 338/338 [00:00<00:00, 622.17it/s]
Loading LoRA adapter: /root/MLP/outputs/lora_billsum_qwen/final_adapter
Loaded 2,905 test records
The following generation flags are not v

## 2. BillSum LoRA — Llama

In [22]:
import os, getpass
from huggingface_hub import login
login()
print("HuggingFace 登录成功")


HuggingFace 登录成功


In [23]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/lora_billsum_llama.yaml 2>&1 | tee logs/lora_billsum_llama.log
!{sys.executable} -u src/evaluate/inference.py --config configs/lora_billsum_llama.yaml --split test_us
!{sys.executable} -u src/evaluate/inference.py --config configs/lora_billsum_llama.yaml --split test_ca
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/lora_billsum_llama/predictions_test_us.jsonl --output outputs/lora_billsum_llama/eval_test_us.json
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/lora_billsum_llama/predictions_test_ca.jsonl --output outputs/lora_billsum_llama/eval_test_ca.json
print("完成")
!cat outputs/lora_billsum_llama/eval_test_us.json


/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/lora_billsum_llama.yaml
Model   : meta-llama/Llama-3.2-1B-Instruct
Train   : /root/MLP/data/billsum/train_sft.jsonl  (15,988 records)
Val     : /root/MLP/data/billsum/val_sft.jsonl
Max input length : 2048
Max output length: 512
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 333.51it/s]
trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750

Preparing datasets...
Tokenizing: 10

## 3. BillSum Full FT — Qwen

In [24]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/full_billsum_qwen.yaml 2>&1 | tee logs/full_billsum_qwen.log
!{sys.executable} -u src/evaluate/inference.py --config configs/full_billsum_qwen.yaml --split test_us
!{sys.executable} -u src/evaluate/inference.py --config configs/full_billsum_qwen.yaml --split test_ca
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/full_billsum_qwen/predictions_test_us.jsonl --output outputs/full_billsum_qwen/eval_test_us.json
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/full_billsum_qwen/predictions_test_ca.jsonl --output outputs/full_billsum_qwen/eval_test_ca.json
print("完成")
!cat outputs/full_billsum_qwen/eval_test_us.json


/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/full_billsum_qwen.yaml
Model   : Qwen/Qwen2.5-1.5B-Instruct
Train   : /root/MLP/data/billsum/train_sft.jsonl  (15,988 records)
Val     : /root/MLP/data/billsum/val_sft.jsonl
Max input length : 2048
Max output length: 512
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 600.13it/s]
Full fine-tuning: 1.54B trainable parameters

Preparing datasets...
Tokenizing: 100%|██████████| 842/842 [00:06<00:00, 138.

## 4. BillSum Full FT — Llama

In [25]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/full_billsum_llama.yaml 2>&1 | tee logs/full_billsum_llama.log
!{sys.executable} -u src/evaluate/inference.py --config configs/full_billsum_llama.yaml --split test_us
!{sys.executable} -u src/evaluate/inference.py --config configs/full_billsum_llama.yaml --split test_ca
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/full_billsum_llama/predictions_test_us.jsonl --output outputs/full_billsum_llama/eval_test_us.json
!{sys.executable} -u src/evaluate/eval_billsum.py --predictions outputs/full_billsum_llama/predictions_test_ca.jsonl --output outputs/full_billsum_llama/eval_test_ca.json
print("完成")
!cat outputs/full_billsum_llama/eval_test_us.json


/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/full_billsum_llama.yaml
Model   : meta-llama/Llama-3.2-1B-Instruct
Train   : /root/MLP/data/billsum/train_sft.jsonl  (15,988 records)
Val     : /root/MLP/data/billsum/val_sft.jsonl
Max input length : 2048
Max output length: 512
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 331.15it/s]
Full fine-tuning: 1.24B trainable parameters

Preparing datasets...
Tokenizing: 100%|██████████| 842/842 [00:05<00:0

## 5. CaseHOLD LoRA — Qwen

In [31]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/lora_casehold_qwen.yaml 2>&1 | tee logs/lora_casehold_qwen.log
!{sys.executable} -u src/evaluate/inference.py --config configs/lora_casehold_qwen.yaml --split test
!{sys.executable} -u src/evaluate/eval_casehold.py --predictions outputs/lora_casehold_qwen/predictions_test.jsonl --output outputs/lora_casehold_qwen/eval_test.json
print("完成")
!cat outputs/lora_casehold_qwen/eval_test.json


/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/lora_casehold_qwen.yaml
Model   : Qwen/Qwen2.5-1.5B-Instruct
Train   : /root/MLP/data/casehold/train_mc.jsonl  (30,652 records)
Val     : /root/MLP/data/casehold/validation_mc.jsonl
Max input length : 1024
Max output length: 256
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 676.22it/s]
trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815

Preparing datasets...
Tokenizing: 1

## 6. CaseHOLD LoRA — Llama

In [32]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train.py --config configs/lora_casehold_llama.yaml 2>&1 | tee logs/lora_casehold_llama.log
!{sys.executable} -u src/evaluate/inference.py --config configs/lora_casehold_llama.yaml --split test
!{sys.executable} -u src/evaluate/eval_casehold.py --predictions outputs/lora_casehold_llama/predictions_test.jsonl --output outputs/lora_casehold_llama/eval_test.json
print("完成")
!cat outputs/lora_casehold_llama/eval_test.json


/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
Config  : configs/lora_casehold_llama.yaml
Model   : meta-llama/Llama-3.2-1B-Instruct
Train   : /root/MLP/data/casehold/train_mc.jsonl  (30,652 records)
Val     : /root/MLP/data/casehold/validation_mc.jsonl
Max input length : 1024
Max output length: 256
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 334.90it/s]
trainable params: 3,407,872 || all params: 1,239,222,272 || trainable%: 0.2750

Preparing datasets...
Tokeni

## 7. CaseHOLD QLoRA — Qwen

In [39]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train_casehold_lora.py --config configs/qlora_casehold_qwen.yaml 2>&1 | tee logs/qlora_casehold_qwen.log
!{sys.executable} -u src/evaluate/inference.py --config configs/qlora_casehold_qwen.yaml --split test
!{sys.executable} -u src/evaluate/eval_casehold.py --predictions outputs/qlora_casehold_qwen/predictions_test.jsonl --output outputs/qlora_casehold_qwen/eval_test.json
print("完成")
!cat outputs/qlora_casehold_qwen/eval_test.json


/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 338/338 [00:00<00:00, 508.88it/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.
]11;?w

## 8. CaseHOLD QLoRA — Llama

In [40]:
import sys, os
os.chdir('/root/MLP')
os.environ['WANDB_MODE'] = 'offline'
!{sys.executable} -u src/train/train_casehold_lora.py --config configs/qlora_casehold_llama.yaml 2>&1 | tee logs/qlora_casehold_llama.log
!{sys.executable} -u src/evaluate/inference.py --config configs/qlora_casehold_llama.yaml --split test
!{sys.executable} -u src/evaluate/eval_casehold.py --predictions outputs/qlora_casehold_llama/predictions_test.jsonl --output outputs/qlora_casehold_llama/eval_test.json
print("完成")
!cat outputs/qlora_casehold_llama/eval_test.json


/root/miniconda3/lib/python3.10/site-packages/torchvision/io/image.py:13: UserWarning: Failed to load image Python extension: 'Could not load this library: /root/miniconda3/lib/python3.10/site-packages/torchvision/image.so'If you don't plan on using image functionality from `torchvision.io`, you can ignore this warning. Otherwise, there might be something wrong with your environment. Did you have `libjpeg` or `libpng` installed before building `torchvision` from source?
  warn(
[INFO] Running in WANDB offline mode
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 146/146 [00:00<00:00, 278.90it/s]
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.
]11;?

## 汇总所有结果

In [41]:
import json, os

results = [
    ('lora_billsum_qwen',   'outputs/lora_billsum_qwen/eval_test_us.json'),
    ('lora_billsum_llama',  'outputs/lora_billsum_llama/eval_test_us.json'),
    ('full_billsum_qwen',   'outputs/full_billsum_qwen/eval_test_us.json'),
    ('full_billsum_llama',  'outputs/full_billsum_llama/eval_test_us.json'),
    ('lora_casehold_qwen',  'outputs/lora_casehold_qwen/eval_test.json'),
    ('lora_casehold_llama', 'outputs/lora_casehold_llama/eval_test.json'),
    ('qlora_casehold_qwen', 'outputs/qlora_casehold_qwen/eval_test.json'),
    ('qlora_casehold_llama','outputs/qlora_casehold_llama/eval_test.json'),
]

print(f'{"实验":<25} {"指标":<15} {"值":>8}')
print('-' * 52)
for name, path in results:
    if not os.path.exists(path):
        print(f'{name:<25} {"未完成":<15}')
        continue
    with open(path) as f:
        d = json.load(f)
    # BillSum 看 rouge2，CaseHOLD 看 accuracy
    if 'rouge2' in d:
        val = d["rouge2"]["mean"] if isinstance(d["rouge2"], dict) else d["rouge2"]
        print(f'{name:<25} {"rouge2":<15} {val:>8.4f}')
    elif 'accuracy' in d:
        val = d["accuracy"]["mean"] if isinstance(d["accuracy"], dict) else d["accuracy"]
        print(f'{name:<25} {"accuracy":<15} {val:>8.4f}')

实验                        指标                     值
----------------------------------------------------
lora_billsum_qwen         rouge2            0.3725
lora_billsum_llama        rouge2            0.3741
full_billsum_qwen         rouge2            0.3688
full_billsum_llama        rouge2            0.3839
lora_casehold_qwen        accuracy          0.8602
lora_casehold_llama       accuracy          0.8617
qlora_casehold_qwen       accuracy          0.7395
qlora_casehold_llama      accuracy          0.7179
